# Efficient Frontier — Baseline Template

**Student Name**: [Your Name]

**Project**: LFC MarketView Efficient Frontier

**Goal**: Use diversification to find the portfolios with the best return for each level of risk

---

## Structure
1. Data, Returns & Correlation (Week 1)
2. The Two-Stock Portfolio (Week 2)
3. Multi-Asset Efficient Frontier (Week 3)

---

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline
print('Libraries loaded')

Libraries loaded


---
## Week 1: Data, Returns & Correlation

**Goal**: Download stock data and describe each stock's risk and return

In [ ]:
# Italian large caps (Borsa Italiana tickers on Yahoo Finance)
TICKERS = {
    'ENI.MI':  'Eni',
    'ISP.MI':  'Intesa Sanpaolo',
    'UCG.MI':  'UniCredit',
    'ENEL.MI': 'Enel',
    'RACE.MI': 'Ferrari',
    'G.MI':    'Generali',
}
BENCHMARK = 'FTSEMIB.MI'

START_DATE = '2019-01-01'
END_DATE   = '2026-10-01'
RISK_FREE  = 0.02            # 2% yearly risk-free rate
TRADING_DAYS = 252

print(f'Stocks: {len(TICKERS)}')
print(f'Date range: {START_DATE} to {END_DATE}')

Stocks: 6
Date range: 2019-01-01 to 2026-10-01


In [ ]:
# Download all tickers
price_data = {}

for ticker in list(TICKERS) + [BENCHMARK]:
    temp = yf.download(ticker, start=START_DATE, end=END_DATE,
                       progress=False, auto_adjust=True)
    if not temp.empty:
        if isinstance(temp.columns, pd.MultiIndex):
            price_data[ticker] = temp['Close'][ticker]
        else:
            price_data[ticker] = temp['Close']
        print(f'YES {ticker}')

all_prices = pd.DataFrame(price_data).dropna()
prices    = all_prices[list(TICKERS)]
benchmark = all_prices[BENCHMARK]

print(f'\nShape: {prices.shape}')
prices.head()

YES ENI.MI
YES ISP.MI
YES UCG.MI
YES ENEL.MI
YES RACE.MI
YES G.MI
YES FTSEMIB.MI

Shape: (1968, 6)


,ENI.MI,ISP.MI,UCG.MI,ENEL.MI,RACE.MI,G.MI
Date,,,,,,
2019-01-02,8.403577,1.115285,7.255135,3.250721,81.466850,9.331881
2019-01-03,8.440168,1.112360,7.152678,3.252000,80.600578,9.357615
2019-01-04,8.745086,1.169575,7.578839,3.296758,84.178642,9.476634
2019-01-07,8.668247,1.189349,7.779297,3.295480,85.986496,9.447683
2019-01-08,8.614582,1.175601,7.651597,3.309546,88.641800,9.441250


In [ ]:
# Plot normalized prices (every stock starts at 1)
normalized = prices / prices.iloc[0]

plt.figure(figsize=(14, 6))
for ticker in normalized.columns:
    plt.plot(normalized.index, normalized[ticker], label=TICKERS[ticker], linewidth=1.5)
plt.title('Normalized Prices (Start = 1)')
plt.ylabel('Growth of €1')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Daily returns -> annualized return and volatility
returns = prices.pct_change().dropna()

annual_return = returns.mean() * TRADING_DAYS
annual_vol    = returns.std() * np.sqrt(TRADING_DAYS)

stats = pd.DataFrame({
    'Name':          [TICKERS[t] for t in returns.columns],
    'Annual Return': annual_return.round(4),
    'Annual Vol':    annual_vol.round(4),
})
stats

,Name,Annual Return,Annual Vol
ENI.MI,Eni,0.1752,0.2827
ISP.MI,Intesa Sanpaolo,0.2705,0.2873
UCG.MI,UniCredit,0.3844,0.3790
ENEL.MI,Enel,0.1544,0.2328
RACE.MI,Ferrari,0.2251,0.2807
G.MI,Generali,0.2170,0.2089


In [ ]:
# TODO (Student): Correlation heatmap
#
# - corr = returns.corr()
# - Plot it with sns.heatmap(corr, annot=True, cmap='coolwarm')
# - Which two stocks are MOST correlated? Which two are LEAST?
#   Does it make sense given their sectors?

print('Correlation heatmap: Student work here')

---
## Week 2: The Two-Stock Portfolio

**Goal**: See diversification happen with just two assets

**TODO**: Change `STOCK_A` and `STOCK_B` to compare a low- and a high-correlation pair

In [ ]:
STOCK_A, STOCK_B = 'ENI.MI', 'RACE.MI'

ret_A, ret_B = annual_return[STOCK_A], annual_return[STOCK_B]
vol_A, vol_B = annual_vol[STOCK_A],    annual_vol[STOCK_B]
corr_AB = returns[STOCK_A].corr(returns[STOCK_B])

weights = np.linspace(0, 1, 101)    # 0%, 1%, ..., 100% in stock A
port_returns, port_vols = [], []

for w in weights:
    r = w * ret_A + (1 - w) * ret_B
    v = np.sqrt(w**2 * vol_A**2 + (1 - w)**2 * vol_B**2
                + 2 * w * (1 - w) * corr_AB * vol_A * vol_B)
    port_returns.append(r)
    port_vols.append(v)

plt.figure(figsize=(10, 6))
plt.plot(port_vols, port_returns, linewidth=2)
plt.scatter([vol_A, vol_B], [ret_A, ret_B], color='red', zorder=3)
plt.annotate(f'100% {TICKERS[STOCK_A]}', (vol_A, ret_A), xytext=(5, 5), textcoords='offset points')
plt.annotate(f'100% {TICKERS[STOCK_B]}', (vol_B, ret_B), xytext=(5, 5), textcoords='offset points')
plt.title(f'{TICKERS[STOCK_A]} / {TICKERS[STOCK_B]} (correlation = {corr_AB:.2f})')
plt.xlabel('Volatility (risk)')
plt.ylabel('Expected return')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TODO (Student): Minimum-variance portfolio
#
# - Find the weight with the lowest volatility: np.argmin(port_vols)
# - Print the weights of A and B and the portfolio's return and volatility
# - Mark it on the chart above with a star (marker='*', s=300)
#
# Then repeat Week 2 with a HIGHLY correlated pair (e.g. ISP.MI / UCG.MI).
# How does the shape of the curve change?

print('Minimum-variance portfolio: Student work here')

---
## Week 3: Multi-Asset Efficient Frontier

**Goal**: Simulate thousands of portfolios from all 6 stocks and find the best ones

In [ ]:
N_PORTFOLIOS = 5_000
rng = np.random.default_rng(42)

cov_matrix = returns.cov() * TRADING_DAYS     # annualized covariance matrix
n_assets = len(TICKERS)

all_weights = np.zeros((N_PORTFOLIOS, n_assets))
results = np.zeros((N_PORTFOLIOS, 3))         # return, volatility, sharpe

for i in range(N_PORTFOLIOS):
    w = rng.random(n_assets)
    w = w / w.sum()                            # weights must add up to 1
    all_weights[i] = w

    port_ret = w @ annual_return.values
    port_vol = np.sqrt(w.T @ cov_matrix.values @ w)
    results[i] = [port_ret, port_vol, (port_ret - RISK_FREE) / port_vol]

frontier = pd.DataFrame(results, columns=['Return', 'Volatility', 'Sharpe'])

plt.figure(figsize=(12, 7))
sc = plt.scatter(frontier['Volatility'], frontier['Return'],
                 c=frontier['Sharpe'], cmap='viridis', s=8)
plt.colorbar(sc, label='Sharpe ratio')
plt.title(f'Efficient Frontier — {N_PORTFOLIOS:,} Random Portfolios')
plt.xlabel('Volatility (risk)')
plt.ylabel('Expected return')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TODO (Student): Find the optimal portfolios
#
# - Max-Sharpe portfolio:   frontier['Sharpe'].idxmax()
# - Min-variance portfolio: frontier['Volatility'].idxmin()
# - Print the weights of each (all_weights[index]) next to the stock names
# - Mark both on the scatter plot with stars
# - Plot the weights of each as a bar chart
#
# Are the weights sensible? Would you put that much in one stock?

print('Optimal portfolios: Student work here')

In [ ]:
# TODO (Student): Growth of €10,000 — Max Sharpe vs Equal Weight vs FTSE MIB
#
# daily portfolio return = returns @ weights
# growth = 10000 * (1 + daily_portfolio_returns).cumprod()

bench_returns = benchmark.pct_change().dropna()
bench_growth  = 10000 * (1 + bench_returns).cumprod()

print('Add your portfolios to the chart below')

# Placeholder chart
plt.figure(figsize=(14, 6))
plt.plot(bench_growth.index, bench_growth, label='FTSE MIB Benchmark', linewidth=2)
plt.title('Growth of €10,000 — Optimal Portfolios vs FTSE MIB')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (€)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# TODO (Student): Performance table (fill in your results)
performance = pd.DataFrame({
    'Metric':        ['Annual Return', 'Volatility', 'Sharpe', 'Max Drawdown'],
    'Max Sharpe':    ['TBD', 'TBD', 'TBD', 'TBD'],
    'Min Variance':  ['TBD', 'TBD', 'TBD', 'TBD'],
    'Equal Weight':  ['TBD', 'TBD', 'TBD', 'TBD'],
    'FTSE MIB':      ['TBD', 'TBD', 'TBD', 'TBD'],
})
performance

,Metric,Max Sharpe,Min Variance,Equal Weight,FTSE MIB
0,Annual Return,TBD,TBD,TBD,TBD
1,Volatility,TBD,TBD,TBD,TBD
2,Sharpe,TBD,TBD,TBD,TBD
3,Max Drawdown,TBD,TBD,TBD,TBD


---
## Conclusions

**TODO**: Write your analysis here

### Best Portfolio
[Answer]

### What Diversification Did
[Answer]

### Why "Optimal" Weights Can Be Misleading
[Answer]

### Limitations
[Answer]

### Potential Improvements
[Answer]

---
**Completed by**: [Your Name] | **Date**: [Date]